# Free embeddins and costum embeddings

In [2]:
import lancedb
import pandas as pd

df = pd.read_json("data/jokes.json")

db = lancedb.connect("knowledge_base")
df.head(), db

(                                               jokes
 0  Parallel lines have so much in common—it’s sad...
 1  ETL stands for “Extract, Transform, Leave for ...
 2  What do you call a snake that runs your script...
 3  Gold walks into a bar. The bartender says, “Au...
 4   C# devs don’t argue; they just throw exceptions.,
 LanceDBConnection(uri='c:\\Users\\lisay\\Documents\\8.Ai_ops\\ai_ops_course\\09_lancedb\\knowledge_base'))

## Sentence transformer
It will download the model first time you use it

In [4]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

model = (
    get_registry().get("sentence-transformers")
    .create(name="intfloat/multilingual-e5-small", device="cpu")
)

class JokeModel(LanceModel):
    jokes: str = model.SourceField() # marks text as input to embedding function
    vector: Vector(model.ndims()) = (model.VectorField()) # vector is destination of computed embedding

table = db.create_table("jokes_sentence_transformer", schema = JokeModel)
table.add(df)

c:\Users\lisay\Documents\8.Ai_ops\ai_ops_course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\lisay\Documents\8.Ai_ops\ai_ops_course\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lisay\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python a

AddResult(version=2)

In [5]:
table.search("why do mathematicians have problems?").to_pandas()

,jokes,vector,_distance
0,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.273110
1,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.281742
2,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.287475
3,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.300044
4,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.310816
5,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0...",0.316030
6,Why did the C# developer go broke? He kept usi...,"[0.020794, 0.007793, -0.040906, -0.02401, 0.03...",0.318841
7,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0....",0.326532
8,"Data engineer motto: If it works, don’t touch ...","[0.003134, -0.036774, -0.050577, -0.060755, 0....",0.331513
9,Why did the Python programmer get bitten? Beca...,"[0.029594, 0.007605, -0.018481, -0.004575, 0.0...",0.334075


In [6]:
table.search("kemi skämt").to_pandas()

,jokes,vector,_distance
0,Chemists have all the solutions… mostly in bea...,"[0.025907, -0.022183, -0.06204, -0.036586, 0.0...",0.357881
1,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.361318
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.372850
3,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.374648
4,"My pipeline failed again, so I’m logging out—l...","[0.014553, -0.021881, 0.002843, -0.057108, 0.0...",0.391870
5,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0...",0.398163
6,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.405061
7,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.407789
8,My Python code works on the first try… said no...,"[0.038937, -0.032795, -0.005337, -0.001472, 0....",0.408450
9,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.411249


## Custom embedder
I want to use from nvidia, which exists in the documentation, but unfortunately it is not implemented. That's why I will implement my own custom embedder for nvidia nemotron

In [7]:
from lancedb.pydantic import LanceModel, Vector
from custom_embeddings.nvidia_nemotron import NvidiaNemotronEmbedding
from lancedb.embeddings import get_registry

embedding_model = get_registry().get("nvidia-nemotron").create()
table_jokes = db.create_table("jokes_nvidia", schema=JokeModel, mode="overwrite")

table_jokes.add(df)


AddResult(version=2)

In [8]:
table_jokes.search("kemi").to_pandas()

,jokes,vector,_distance
0,Chemists have all the solutions… mostly in bea...,"[0.025907, -0.022183, -0.06204, -0.036586, 0.0...",0.337850
1,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.360644
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.370747
3,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.387295
4,Never trust an atom—they make up everything.,"[0.059282, -0.023624, -0.089997, -0.071139, 0....",0.394323
5,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0...",0.409014
6,I asked the data lake if it had my file. It sa...,"[0.069376, -0.0138, -0.052487, -0.050929, 0.08...",0.411283
7,Python is so friendly even whitespace gets a say.,"[0.020338, 0.010201, -0.026083, -0.032346, 0.0...",0.418516
8,"My pipeline failed again, so I’m logging out—l...","[0.014553, -0.021881, 0.002843, -0.057108, 0.0...",0.419971
9,My Python code works on the first try… said no...,"[0.038937, -0.032795, -0.005337, -0.001472, 0....",0.421265


In [9]:
table_jokes.search("why do mathematicians have problems?").to_pandas()

,jokes,vector,_distance
0,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.273110
1,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.281742
2,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.287475
3,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.300044
4,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.310816
5,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0...",0.316030
6,Why did the C# developer go broke? He kept usi...,"[0.020794, 0.007793, -0.040906, -0.02401, 0.03...",0.318841
7,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0....",0.326532
8,"Data engineer motto: If it works, don’t touch ...","[0.003134, -0.036774, -0.050577, -0.060755, 0....",0.331513
9,Why did the Python programmer get bitten? Beca...,"[0.029594, 0.007605, -0.018481, -0.004575, 0.0...",0.334075
